## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)

In [1]:
## Only uncomment and run in colab or kaggle to install packages
# from IPython.display import clear_output
# !pip install uv
# !uv pip install "polars[gpu]>=1.27.0" --extra-index-url=https://pypi.nvidia.com
# clear_output()

In [2]:
import polars as pl

In [3]:
zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}

df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [4]:
df.head().collect(engine="streaming")

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year,month,PU_Borough,PU_Zone,PU_service_zone,DO_Borough,DO_Zone,DO_service_zone
i16,datetime[ms],datetime[ms],u16,f32,u8,str,u16,u16,u8,f32,f32,f32,f32,f32,f32,f32,f32,f32,i16,i8,str,str,str,str,str,str
2,2014-01-05 07:07:00,2014-01-05 07:30:00,1,16.1,2,null,170,132,1,52.0,0.0,0.5,11.47,5.33,0.0,69.300003,null,null,2014,1,"""Manhattan""","""Murray Hill""","""Yellow Zone""","""Queens""","""JFK Airport""","""Airports"""
2,2014-01-05 07:53:00,2014-01-05 08:32:00,2,18.67,2,null,24,132,2,52.0,0.0,0.5,0.0,5.33,0.0,57.830002,null,null,2014,1,"""Manhattan""","""Bloomingdale""","""Yellow Zone""","""Queens""","""JFK Airport""","""Airports"""
1,2014-01-05 07:35:04,2014-01-05 08:09:47,1,16.200001,2,"""N""",170,132,1,52.0,0.0,0.5,11.56,5.33,0.0,69.389999,null,null,2014,1,"""Manhattan""","""Murray Hill""","""Yellow Zone""","""Queens""","""JFK Airport""","""Airports"""
1,2014-01-05 07:46:03,2014-01-05 07:48:32,1,0.7,1,"""N""",24,238,1,4.5,0.0,0.5,0.0,0.0,0.0,5.0,null,null,2014,1,"""Manhattan""","""Bloomingdale""","""Yellow Zone""","""Manhattan""","""Upper West Side North""","""Yellow Zone"""
1,2014-01-05 07:29:41,2014-01-05 07:41:40,1,3.9,1,"""N""",170,238,1,13.5,0.0,0.5,2.8,0.0,0.0,16.799999,null,null,2014,1,"""Manhattan""","""Murray Hill""","""Yellow Zone""","""Manhattan""","""Upper West Side North""","""Yellow Zone"""


## Streaming Engine

In [5]:
%%time
df.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine="streaming")

CPU times: user 1min 22s, sys: 1.45 s, total: 1min 24s
Wall time: 3.67 s


trip_distance,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,u32,f32,f32,f32,f32
-4.0840124e7,1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,1,2.5,2.5,-3.4467e6,0.0
-186318.40625,1,2.5,2.5,-186318.40625,0.0
-37264.53125,1,25.809999,25.809999,-37264.53125,0.0


## GPU in memory

In [6]:
gpu_in_mem_engine = pl.GPUEngine(device=0, raise_on_fail=True)

In [7]:
%%time
# This will fail if GPU memory is not large enough
df.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine=gpu_in_mem_engine)

CPU times: user 8.41 s, sys: 895 ms, total: 9.3 s
Wall time: 5.71 s


trip_distance,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,u32,f32,f32,f32,f32
-4.0840124e7,1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,1,2.5,2.5,-3.4467e6,0.0
-186318.40625,1,2.5,2.5,-186318.40625,0.0
-37264.53125,1,25.809999,25.809999,-37264.53125,0.0


In [8]:
%%time
res, prof = (
    df_green.group_by("trip_distance")
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort("trip_distance")
    .head()
    .profile(engine="streaming")
)

CPU times: user 16.9 s, sys: 2.12 s, total: 19.1 s
Wall time: 1.59 s


In [9]:
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

node,start,end,pct_time_spent
str,u64,u64,f64
"""optimization""",0,122611,7.734109
"""simple-projection(LocationID)""",122611,122613,0.000126
"""simple-projection(LocationID)""",122635,122642,0.000442
"""simple-projection(LocationID)""",122638,122641,0.000189
"""join(PULocationID)""",122649,856717,46.303856
"""simple-projection(DOLocationID…",857649,857654,0.000315
"""join(DOLocationID)""",857676,1058604,12.674223
"""simple-projection(trip_distanc…",1058617,1058623,0.000378
"""group_by_partitioned(trip_dist…",1058631,1584806,33.190292


## GPU in memory. More complex queries

In [10]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine="streaming")

CPU times: user 3min 26s, sys: 10.5 s, total: 3min 36s
Wall time: 9.97 s


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-4.0840124e7,"""Queens""","""Queens""",1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,"""Manhattan""","""Manhattan""",1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,"""Queens""","""Manhattan""",1,2.5,2.5,-3.4467e6,0.0
-186318.40625,"""Queens""","""Queens""",1,2.5,2.5,-186318.40625,0.0
-37264.53125,"""Brooklyn""","""Queens""",1,25.809999,25.809999,-37264.53125,0.0


In [11]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_in_mem_engine)

CPU times: user 6.83 s, sys: 757 ms, total: 7.59 s
Wall time: 5.52 s


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-4.0840124e7,"""Queens""","""Queens""",1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,"""Manhattan""","""Manhattan""",1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,"""Queens""","""Manhattan""",1,2.5,2.5,-3.4467e6,0.0
-186318.40625,"""Queens""","""Queens""",1,2.5,2.5,-186318.40625,0.0
-37264.53125,"""Brooklyn""","""Queens""",1,25.809999,25.809999,-37264.53125,0.0


## UVM for single GPU
[UVM, data larger than GPU memory post (NVIDIA post)](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)

[UVM, data larger than GPU memory post (Polars post)](https://pola.rs/posts/uvm-larger-than-ram-gpu/)


In [5]:
import rmm

mr = rmm.mr.PrefetchResourceAdaptor(rmm.mr.PoolMemoryResource(rmm.mr.ManagedMemoryResource()))
gpu_uvm_engine = pl.GPUEngine(memory_resource=mr, raise_on_fail=True)

In [6]:
%%time
df.with_columns().group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).collect(engine=gpu_uvm_engine)

CPU times: user 6.52 s, sys: 736 ms, total: 7.25 s
Wall time: 5.81 s


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-4.0840124e7,"""Queens""","""Queens""",1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,"""Manhattan""","""Manhattan""",1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,"""Queens""","""Manhattan""",1,2.5,2.5,-3.4467e6,0.0
-186318.40625,"""Queens""","""Queens""",1,2.5,2.5,-186318.40625,0.0
-37264.53125,"""Brooklyn""","""Queens""",1,25.809999,25.809999,-37264.53125,0.0
…,…,…,…,…,…,…,…
1.9072628e7,"""Manhattan""","""Manhattan""",1,2.5,2.5,1.9072628e7,0.0
2.062186e7,"""Manhattan""","""Manhattan""",1,2.5,2.5,2.062186e7,0.0
4.2107492e7,"""Manhattan""","""Manhattan""",1,2.5,2.5,4.2107492e7,0.0


## GPU Streaming
[Polars GPU Streaming NVIDIA blog post](https://developer.nvidia.com/blog/rapids-adds-gpu-polars-streaming-a-unified-gnn-api-and-zero-code-ml-speedups)

In [14]:
gpu_streaming_engine = pl.GPUEngine(executor="streaming", raise_on_fail=True)

In [15]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_streaming_engine)

CPU times: user 15.2 s, sys: 9.89 s, total: 25.1 s
Wall time: 15.2 s


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-4.0840124e7,"""Queens""","""Queens""",1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,"""Manhattan""","""Manhattan""",1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,"""Queens""","""Manhattan""",1,2.5,2.5,-3.4467e6,0.0
-186318.40625,"""Queens""","""Queens""",1,2.5,2.5,-186318.40625,0.0
-37264.53125,"""Brooklyn""","""Queens""",1,25.809999,25.809999,-37264.53125,0.0


In [16]:
## GPU Streaming